In [1]:
# Verify repaired dashboard dependencies

import numpy as np
import pandas as pd
import pyarrow
import numexpr
import bottleneck
import xarray
import streamlit
import plotly
import plotly.express as px

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Streamlit:", streamlit.__version__)
print("Plotly:", plotly.__version__)

print("\nAll dashboard dependencies imported successfully.")

NumPy: 1.26.4
Pandas: 2.2.3
Streamlit: 1.66.1
Plotly: 7.1.0

All dashboard dependencies imported successfully.


In [3]:
#  Streamlit Interactive Dashboard
# Creates all dashboard files, validates dependencies, and launches the application.

from pathlib import Path
import sys
import subprocess
import importlib
import time

# ---------------------------------------------------------
# Project configuration
# ---------------------------------------------------------

PROJECT_PATH = Path(
    r"C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy"
)

NOTEBOOK_PATH = PROJECT_PATH / "notebooks"
DASHBOARD_PATH = PROJECT_PATH / "dashboard"
DATASET_PATH = (
    PROJECT_PATH / "data" / "processed"
    / "European_Bank_Cleaned_Analytical.csv"
)

DASHBOARD_PATH.mkdir(parents=True, exist_ok=True)

if not PROJECT_PATH.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_PATH}")

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Processed dataset not found: {DATASET_PATH}")

# ---------------------------------------------------------
# Install and import required dependencies
# ---------------------------------------------------------

required_packages = {
    "streamlit": "streamlit",
    "pandas": "pandas",
    "numpy": "numpy",
    "plotly": "plotly"
}

missing_packages = []

for package, module in required_packages.items():
    try:
        importlib.import_module(module)
        print(f"{package}: Available")
    except ImportError:
        missing_packages.append(package)

if missing_packages:
    print("Installing missing dependencies:", missing_packages)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", *missing_packages
    ])

import pandas as pd
import numpy as np
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

print("All dashboard dependencies are ready.")

# ---------------------------------------------------------
# Create config.py
# ---------------------------------------------------------

config_code = r'''
from pathlib import Path

PROJECT_PATH = Path(__file__).resolve().parents[1]
DATASET_PATH = (
    PROJECT_PATH / "data" / "processed"
    / "European_Bank_Cleaned_Analytical.csv"
)

APP_TITLE = "Customer Engagement & Product Utilization Analytics"
APP_ICON = "📊"
LAYOUT = "wide"

HIGH_BALANCE_PERCENTILE = 0.75
'''

# ---------------------------------------------------------
# Create data_loader.py
# ---------------------------------------------------------

data_loader_code = r'''
import pandas as pd
import numpy as np
import streamlit as st
from config import DATASET_PATH

ALIASES = {
    "customer_id": ["CustomerId", "CustomerID", "customer_id", "Customer Id"],
    "geography": ["Geography", "geography", "Country"],
    "gender": ["Gender", "gender"],
    "age": ["Age", "age"],
    "tenure": ["Tenure", "tenure"],
    "balance": ["Balance", "balance"],
    "products": ["NumOfProducts", "NumberOfProducts", "num_products"],
    "active": ["IsActiveMember", "ActiveMember", "is_active_member"],
    "card": ["HasCrCard", "HasCreditCard", "has_credit_card"],
    "exited": ["Exited", "Churn", "churn", "CustomerExited"],
    "credit_score": ["CreditScore", "credit_score"],
    "salary": ["EstimatedSalary", "Salary", "estimated_salary"],
    "surname": ["Surname", "surname"]
}

def find_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

@st.cache_data
def load_data():
    df = pd.read_csv(DATASET_PATH)

    for standard, candidates in ALIASES.items():
        found = find_column(df, candidates)
        if found and found != standard:
            df[standard] = df[found]

    numeric_columns = [
        "age", "tenure", "balance", "products", "active",
        "card", "exited", "credit_score", "salary"
    ]

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "customer_id" not in df.columns:
        df["customer_id"] = np.arange(1, len(df) + 1)

    if "active" not in df.columns and "activity_status" in df.columns:
        df["active"] = (
            df["activity_status"].astype(str).str.lower()
            .map({"active": 1, "inactive": 0})
        )

    if "products" not in df.columns:
        df["products"] = 0

    if "active" not in df.columns:
        df["active"] = 0

    if "exited" not in df.columns:
        df["exited"] = 0

    if "balance" not in df.columns:
        df["balance"] = 0

    df["products"] = df["products"].fillna(0)
    df["active"] = df["active"].fillna(0)
    df["exited"] = df["exited"].fillna(0)
    df["balance"] = df["balance"].fillna(0)

    df["active"] = df["active"].clip(0, 1)
    df["exited"] = df["exited"].clip(0, 1)

    df["activity_status"] = np.where(
        df["active"] == 1, "Active", "Inactive"
    )

    df["customer_status"] = np.where(
        df["exited"] == 1, "Exited", "Retained"
    )

    df["product_count_category"] = np.select(
        [
            df["products"] <= 1,
            df["products"] == 2,
            df["products"] >= 3
        ],
        ["Single Product", "Two Products", "Three or More Products"],
        default="Unknown"
    )

    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[0, 25, 35, 45, 55, 100],
            labels=["18-25", "26-35", "36-45", "46-55", "56+"]
        )

    if "geography" not in df.columns:
        df["geography"] = "Unknown"

    if "gender" not in df.columns:
        df["gender"] = "Unknown"

    positive_balances = df.loc[df["balance"] > 0, "balance"]

    if len(positive_balances):
        threshold = positive_balances.quantile(0.75)
    else:
        threshold = 0

    df["high_balance_flag"] = (
        (df["balance"] >= threshold) & (df["balance"] > 0)
    ).astype(int)

    df["inactive_high_balance_flag"] = (
        (df["active"] == 0) & (df["high_balance_flag"] == 1)
    ).astype(int)

    df["relationship_strength_index"] = (
        60 * df["active"]
        + 40 * np.minimum(df["products"], 2) / 2
    )

    df["relationship_strength_category"] = pd.cut(
        df["relationship_strength_index"],
        bins=[-1, 24, 49, 74, 89, 100],
        labels=[
            "Very Weak", "Weak", "Moderate",
            "Strong", "Very Strong"
        ]
    )

    df["premium_customer_candidate"] = (
        (df["high_balance_flag"] == 1)
        & (df["products"] >= 2)
    ).astype(int)

    df["premium_risk_segment"] = np.select(
        [
            (df["active"] == 0) & (df["products"] <= 1)
            & (df["exited"] == 0),

            (df["active"] == 0) & (df["products"] >= 2)
            & (df["exited"] == 0),

            (df["active"] == 1) & (df["high_balance_flag"] == 1),

            (df["exited"] == 1) & (df["high_balance_flag"] == 1)
        ],
        [
            "Critical Attention - Inactive Single Product",
            "High Attention - Inactive Multi Product",
            "Monitor - Active High Balance",
            "Historical Churn - Exited High Balance"
        ],
        default="Standard Segment"
    )

    return df
'''

# ---------------------------------------------------------
# Create analytics.py
# ---------------------------------------------------------

analytics_code = r'''
import pandas as pd
import numpy as np

def safe_rate(numerator, denominator):
    return (
        float(numerator) / float(denominator) * 100
        if denominator else 0.0
    )

def calculate_kpis(df):
    total = len(df)

    retained = int((df["exited"] == 0).sum())
    exited = int((df["exited"] == 1).sum())

    active = df[df["active"] == 1]
    inactive = df[df["active"] == 0]

    active_retention = safe_rate(
        (active["exited"] == 0).sum(), len(active)
    )

    inactive_retention = safe_rate(
        (inactive["exited"] == 0).sum(), len(inactive)
    )

    engagement_ratio = (
        active_retention / inactive_retention
        if inactive_retention else 0
    )

    product_depth = (
        df["products"].sum() / total if total else 0
    )

    high_balance = df[df["high_balance_flag"] == 1]

    high_balance_disengagement = safe_rate(
        high_balance["inactive_high_balance_flag"].sum(),
        len(high_balance)
    )

    if "card" in df.columns:
        cardholders = df[df["card"] == 1]
        non_cardholders = df[df["card"] == 0]

        card_retention = safe_rate(
            (cardholders["exited"] == 0).sum(), len(cardholders)
        )

        non_card_retention = safe_rate(
            (non_cardholders["exited"] == 0).sum(),
            len(non_cardholders)
        )

        card_stickiness = card_retention - non_card_retention
    else:
        card_stickiness = 0

    return {
        "total_customers": total,
        "retained_customers": retained,
        "exited_customers": exited,
        "retention_rate": safe_rate(retained, total),
        "churn_rate": safe_rate(exited, total),
        "active_customers": len(active),
        "inactive_customers": len(inactive),
        "engagement_retention_ratio": engagement_ratio,
        "product_depth_index": product_depth,
        "high_balance_disengagement_rate": high_balance_disengagement,
        "credit_card_stickiness_score": card_stickiness,
        "average_relationship_strength": (
            df["relationship_strength_index"].mean()
            if total else 0
        )
    }

def group_churn(df, column):
    if column not in df.columns:
        return pd.DataFrame()

    result = (
        df.groupby(column, observed=False)
        .agg(
            customers=("exited", "size"),
            exited=("exited", "sum"),
            average_products=("products", "mean")
        )
        .reset_index()
    )

    result["churn_rate"] = (
        result["exited"] / result["customers"] * 100
    )

    return result

def validate_kpis(df, kpis):
    checks = {
        "Total customers": kpis["total_customers"] == len(df),
        "Retained + exited": (
            kpis["retained_customers"] + kpis["exited_customers"]
            == len(df)
        ),
        "Active + inactive": (
            kpis["active_customers"] + kpis["inactive_customers"]
            == len(df)
        ),
        "Product depth": np.isclose(
            kpis["product_depth_index"],
            df["products"].sum() / len(df) if len(df) else 0
        )
    }

    return checks
'''

# ---------------------------------------------------------
# Create components.py
# ---------------------------------------------------------

components_code = r'''
import streamlit as st
import pandas as pd

def render_header():
    st.title("📊 Customer Engagement & Product Utilization Analytics")
    st.caption(
        "Customer retention, engagement, product adoption and premium risk analysis"
    )

def render_kpis(kpis):
    columns = st.columns(4)

    metrics = [
        ("Total Customers", f'{kpis["total_customers"]:,}'),
        ("Retention Rate", f'{kpis["retention_rate"]:.2f}%'),
        ("Churn Rate", f'{kpis["churn_rate"]:.2f}%'),
        ("Product Depth Index", f'{kpis["product_depth_index"]:.2f}')
    ]

    for col, (label, value) in zip(columns, metrics):
        col.metric(label, value)

    columns = st.columns(4)

    metrics = [
        ("Active Customers", f'{kpis["active_customers"]:,}'),
        ("Inactive Customers", f'{kpis["inactive_customers"]:,}'),
        ("High-Balance Disengagement", f'{kpis["high_balance_disengagement_rate"]:.2f}%'),
        ("Relationship Strength", f'{kpis["average_relationship_strength"]:.2f}')
    ]

    for col, (label, value) in zip(columns, metrics):
        col.metric(label, value)

def apply_filters(df):
    st.sidebar.header("Common Dashboard Filters")

    result = df.copy()

    if "geography" in result.columns:
        options = sorted(result["geography"].dropna().astype(str).unique())
        selected = st.sidebar.multiselect(
            "Geography", options, default=options
        )
        result = result[result["geography"].astype(str).isin(selected)]

    if "gender" in result.columns:
        options = sorted(result["gender"].dropna().astype(str).unique())
        selected = st.sidebar.multiselect(
            "Gender", options, default=options
        )
        result = result[result["gender"].astype(str).isin(selected)]

    activity_options = ["Active", "Inactive"]
    selected_activity = st.sidebar.multiselect(
        "Activity Status", activity_options, default=activity_options
    )
    result = result[
        result["activity_status"].isin(selected_activity)
    ]

    status_options = ["Retained", "Exited"]
    selected_status = st.sidebar.multiselect(
        "Customer Status", status_options, default=status_options
    )
    result = result[
        result["customer_status"].isin(selected_status)
    ]

    product_options = sorted(
        result["product_count_category"].dropna().unique().tolist()
    )
    selected_products = st.sidebar.multiselect(
        "Product Category", product_options, default=product_options
    )
    result = result[
        result["product_count_category"].isin(selected_products)
    ]

    if "age" in result.columns and result["age"].notna().any():
        minimum = int(result["age"].min())
        maximum = int(result["age"].max())

        if minimum < maximum:
            age_range = st.sidebar.slider(
                "Age Range",
                minimum,
                maximum,
                (minimum, maximum)
            )
            result = result[
                result["age"].between(age_range[0], age_range[1])
            ]

    return result

def render_customer_table(df):
    st.subheader("Customer Records")

    search = st.text_input(
        "Search customer by ID or surname",
        key="customer_search"
    )

    filtered = df.copy()

    if search.strip():
        mask = filtered["customer_id"].astype(str).str.contains(
            search, case=False, na=False
        )

        if "surname" in filtered.columns:
            mask |= filtered["surname"].astype(str).str.contains(
                search, case=False, na=False
            )

        filtered = filtered[mask]

    st.write(f"Matching customers: {len(filtered):,}")

    st.dataframe(
        filtered,
        use_container_width=True,
        hide_index=True
    )

    csv = filtered.to_csv(index=False).encode("utf-8")

    st.download_button(
        "Download filtered customers as CSV",
        data=csv,
        file_name="filtered_customer_data.csv",
        mime="text/csv"
    )
'''

# ---------------------------------------------------------
# Create app.py
# ---------------------------------------------------------

app_code = r'''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from config import APP_TITLE, APP_ICON, LAYOUT
from data_loader import load_data
from analytics import calculate_kpis, group_churn, validate_kpis
from components import (
    render_header, render_kpis, apply_filters,
    render_customer_table
)

st.set_page_config(
    page_title=APP_TITLE,
    page_icon=APP_ICON,
    layout=LAYOUT
)

df = load_data()

render_header()

st.sidebar.title("Dashboard Navigation")

pages = [
    "Executive Overview",
    "Engagement Analytics",
    "Product Adoption Analytics",
    "Premium Customer Risk Detector",
    "Retention Strength Analytics",
    "Customer Explorer"
]

selected_page = st.sidebar.radio("Select Dashboard Page", pages)

filtered_df = apply_filters(df)

st.sidebar.divider()
st.sidebar.metric("Filtered Customers", f"{len(filtered_df):,}")

if filtered_df.empty:
    st.warning(
        "No customers match the selected filters. "
        "Please change or reset your filters."
    )
    st.stop()

kpis = calculate_kpis(filtered_df)

if selected_page == "Executive Overview":
    st.header("Executive Overview")
    render_kpis(kpis)

    col1, col2 = st.columns(2)

    with col1:
        status = filtered_df["customer_status"].value_counts().reset_index()
        status.columns = ["Customer Status", "Customers"]

        fig = px.pie(
            status,
            names="Customer Status",
            values="Customers",
            title="Customer Retention Distribution",
            hole=0.45
        )
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        activity = filtered_df.groupby(
            "activity_status"
        )["exited"].mean().mul(100).reset_index()

        activity.columns = ["Activity Status", "Churn Rate"]

        fig = px.bar(
            activity,
            x="Activity Status",
            y="Churn Rate",
            title="Churn Rate by Activity Status",
            text_auto=".2f"
        )
        st.plotly_chart(fig, use_container_width=True)

    geography = group_churn(filtered_df, "geography")

    if not geography.empty:
        st.subheader("Geographical Retention Overview")
        fig = px.bar(
            geography,
            x="geography",
            y="churn_rate",
            color="customers",
            title="Geographical Churn Rate",
            text_auto=".2f"
        )
        st.plotly_chart(fig, use_container_width=True)

elif selected_page == "Engagement Analytics":
    st.header("Engagement Analytics")
    render_kpis(kpis)

    engagement = group_churn(filtered_df, "activity_status")

    fig = px.bar(
        engagement,
        x="activity_status",
        y="churn_rate",
        color="customers",
        title="Churn Rate by Engagement Status",
        text_auto=".2f"
    )
    st.plotly_chart(fig, use_container_width=True)

    if "age_group" in filtered_df.columns:
        age_data = group_churn(filtered_df, "age_group")

        fig = px.bar(
            age_data,
            x="age_group",
            y="churn_rate",
            title="Churn Rate by Age Group",
            text_auto=".2f"
        )
        st.plotly_chart(fig, use_container_width=True)

    if "tenure" in filtered_df.columns:
        fig = px.histogram(
            filtered_df,
            x="tenure",
            color="activity_status",
            title="Customer Tenure and Engagement",
            barmode="overlay"
        )
        st.plotly_chart(fig, use_container_width=True)

elif selected_page == "Product Adoption Analytics":
    st.header("Product Adoption Analytics")
    render_kpis(kpis)

    product_data = group_churn(
        filtered_df, "product_count_category"
    )

    col1, col2 = st.columns(2)

    with col1:
        fig = px.bar(
            product_data,
            x="product_count_category",
            y="customers",
            title="Customer Distribution by Product Count",
            text_auto=True
        )
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        fig = px.bar(
            product_data,
            x="product_count_category",
            y="churn_rate",
            title="Churn Rate by Product Category",
            text_auto=".2f"
        )
        st.plotly_chart(fig, use_container_width=True)

    fig = px.box(
        filtered_df,
        x="product_count_category",
        y="relationship_strength_index",
        color="product_count_category",
        title="Relationship Strength by Product Adoption"
    )
    st.plotly_chart(fig, use_container_width=True)

elif selected_page == "Premium Customer Risk Detector":
    st.header("Premium Customer Risk Detector")

    premium = filtered_df[
        filtered_df["high_balance_flag"] == 1
    ]

    st.metric("High-Balance Customers", f"{len(premium):,}")

    risk_counts = (
        filtered_df["premium_risk_segment"]
        .value_counts().reset_index()
    )
    risk_counts.columns = ["Risk Segment", "Customers"]

    fig = px.bar(
        risk_counts,
        x="Risk Segment",
        y="Customers",
        color="Risk Segment",
        title="Premium Customer Risk Segments",
        text_auto=True
    )
    st.plotly_chart(fig, use_container_width=True)

    critical = filtered_df[
        filtered_df["premium_risk_segment"].isin([
            "Critical Attention - Inactive Single Product",
            "High Attention - Inactive Multi Product"
        ])
        & (filtered_df["exited"] == 0)
    ]

    st.subheader("Retained Customers Requiring Attention")
    st.dataframe(critical, use_container_width=True, hide_index=True)

    st.download_button(
        "Export Premium Risk Customers",
        critical.to_csv(index=False).encode("utf-8"),
        "premium_customer_risk.csv",
        "text/csv"
    )

elif selected_page == "Retention Strength Analytics":
    st.header("Retention Strength Analytics")
    render_kpis(kpis)

    strength = (
        filtered_df["relationship_strength_category"]
        .value_counts().reset_index()
    )
    strength.columns = ["Strength Category", "Customers"]

    fig = px.pie(
        strength,
        names="Strength Category",
        values="Customers",
        title="Relationship Strength Distribution",
        hole=0.4
    )
    st.plotly_chart(fig, use_container_width=True)

    fig = px.histogram(
        filtered_df,
        x="relationship_strength_index",
        color="customer_status",
        nbins=20,
        title="Relationship Strength and Customer Status"
    )
    st.plotly_chart(fig, use_container_width=True)

    strength_churn = group_churn(
        filtered_df, "relationship_strength_category"
    )

    fig = px.bar(
        strength_churn,
        x="relationship_strength_category",
        y="churn_rate",
        title="Churn Rate by Relationship Strength",
        text_auto=".2f"
    )
    st.plotly_chart(fig, use_container_width=True)

elif selected_page == "Customer Explorer":
    st.header("Customer Explorer")
    render_kpis(kpis)
    render_customer_table(filtered_df)

st.divider()

with st.expander("KPI Definitions and Validation"):
    st.write(
        "**Engagement Retention Ratio:** Active customer retention rate "
        "divided by inactive customer retention rate."
    )
    st.write(
        "**Product Depth Index:** Average number of products per customer."
    )
    st.write(
        "**High-Balance Disengagement Rate:** Inactive high-balance "
        "customers divided by all high-balance customers."
    )
    st.write(
        "**Credit Card Stickiness Score:** Cardholder retention rate "
        "minus non-cardholder retention rate, in percentage points."
    )
    st.write(
        "**Relationship Strength Index:** 60% activity score and "
        "40% product adoption score, on a 0–100 scale."
    )

    checks = validate_kpis(filtered_df, kpis)

    for name, passed in checks.items():
        if passed:
            st.success(f"{name}: Passed")
        else:
            st.error(f"{name}: Failed")

st.caption(
    "Analytical dashboard based on historical customer data. "
    "Associations do not establish causation or predict future churn."
)
'''

# ---------------------------------------------------------
# Write all five Python files
# ---------------------------------------------------------

dashboard_files = {
    "config.py": config_code,
    "data_loader.py": data_loader_code,
    "analytics.py": analytics_code,
    "components.py": components_code,
    "app.py": app_code
}

for filename, content in dashboard_files.items():
    file_path = DASHBOARD_PATH / filename
    file_path.write_text(content.strip() + "\n", encoding="utf-8")
    print(f"Created: {file_path}")

# ---------------------------------------------------------
# Validate Python syntax
# ---------------------------------------------------------

import ast

print("\nChecking Python file syntax...")

for filename in dashboard_files:
    file_path = DASHBOARD_PATH / filename

    source = file_path.read_text(encoding="utf-8-sig")
    ast.parse(source, filename=filename)

    print(f"{filename}: Syntax OK")

# ---------------------------------------------------------
# Validate processed dataset and dashboard data loading
# ---------------------------------------------------------

print("\nChecking processed dataset...")

dataset = pd.read_csv(DATASET_PATH)

print("Dataset shape:", dataset.shape)
print("Dataset columns:", dataset.columns.tolist())

if dataset.empty:
    raise ValueError("Processed dataset contains no records.")

print("\nDashboard file verification:")
for filename in dashboard_files:
    file_path = DASHBOARD_PATH / filename
    print(f"{filename}: {file_path.stat().st_size:,} bytes")

# ---------------------------------------------------------
# Launch Streamlit dashboard
# ---------------------------------------------------------

print("\nStarting Streamlit dashboard...")

if "_dashboard_process" in globals():
    if _dashboard_process.poll() is None:
        print("A dashboard process is already running.")
    else:
        _dashboard_process = None

if "_dashboard_process" not in globals() or _dashboard_process is None:
    _dashboard_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            str(DASHBOARD_PATH / "app.py"),
            "--server.port",
            "8501"
        ],
        cwd=str(DASHBOARD_PATH)
    )

    time.sleep(5)

    if _dashboard_process.poll() is not None:
        raise RuntimeError(
            "Streamlit stopped unexpectedly. Check the error output."
        )

print("\n" + "=" * 60)
print("STREAMLIT DASHBOARD IMPLEMENTATION COMPLETED")
print("=" * 60)

print("Dashboard folder:", DASHBOARD_PATH)
print("Dataset:", DATASET_PATH)
print("Dashboard URL: http://localhost:8501")
print("Keep the notebook kernel running while using the dashboard.")
print("Stop Streamlit using Ctrl+C in its process terminal.")

streamlit: Available
pandas: Available
numpy: Available
plotly: Available
All dashboard dependencies are ready.
Created: C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\dashboard\config.py
Created: C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\dashboard\data_loader.py
Created: C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\dashboard\analytics.py
Created: C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\dashboard\components.py
Created: C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\dashboard\app.py

Checking Python file syntax...
config.py: Syntax OK
data_loader.py: Syntax OK
analytics.py: Syntax OK
components.py: Syntax OK
app.py: Syntax OK

Checking processed dataset...
Dataset shape: (10000, 26)
Dataset columns: ['year', 'customerid', 'surname', 'creditscore', 'geography', 'gender', 'age',